# Importações 

In [7]:
# Bibliotecas padrão
import os
import gc
import json
import shutil
import zipfile

# Bibliotecas de terceiros
import ee
import geemap
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, silhouette_samples
from sklearn.mixture import GaussianMixture
from shapely.geometry import box
from sklearn.cluster import KMeans
from io import BytesIO
from glob import glob

print("Bibliotecas carregadas")

Bibliotecas carregadas


In [8]:
# Roda o comando no terminal : earthengine authenticate --auth_mode=notebook
ee.Authenticate()
ee.Initialize(project="spatial-yew-490017-r3")
print(ee.String("Hello from the Earth Engine servers!").getInfo())

Hello from the Earth Engine servers!


In [9]:
# mostrar todas as colunas
pd.set_option("display.max_columns", None)

# não quebrar a largura da tabela
pd.set_option("display.expand_frame_repr", False)

# opcional: aumentar a largura máxima exibida
pd.set_option("display.width", 1000)

pd.set_option("display.max_rows", None)  # sem limite

In [10]:
FILES_DIR = "files"
os.makedirs(FILES_DIR, exist_ok=True)

PRINTS_DIR = os.path.join(FILES_DIR, "prints")
os.makedirs(PRINTS_DIR, exist_ok=True)

In [11]:
METRIC_CRS = "EPSG:5880"  # SIRGAS 2000 / Brazil Polyconic

In [12]:
# Limites das UCs federais, via serviço WFS da INDE/ICMBio

ucs_dir = os.path.join(FILES_DIR, "limites_ucs")
os.makedirs(ucs_dir, exist_ok=True)

uc_typename = "ICMBio:limiteucsfederais_a"
uc_output_path = os.path.join(ucs_dir, "limites_ucs.geojson")

if os.path.exists(uc_output_path):
    print(f"Already downloaded: {uc_output_path}")
else:
    wfs_url = (
        "https://geoservicos.inde.gov.br/geoserver/ICMBio/ows"
        f"?service=WFS&version=2.0.0&request=GetFeature"
        f"&typeName={uc_typename}&outputFormat=application/json"
    )
    response = requests.get(wfs_url, timeout=180, verify=False)
    response.raise_for_status()
    with open(uc_output_path, "wb") as f:
        f.write(response.content)
    print(f"Saved: {uc_output_path}")

gdf_ucs = gpd.read_file(uc_output_path)

# Reprojetar para CRS métrico
gdf_ucs = gdf_ucs.to_crs("EPSG:5880")
gdf_ucs["geometry"] = gdf_ucs.geometry.buffer(0)  # corrige geometrias inválidas

Already downloaded: files\limites_ucs\limites_ucs.geojson


# AAF (ICMBio)

### Importação

In [13]:
# AAF — download via ArcGIS FeatureServer (anos 2010-2026)

BASE_URL = "https://services3.arcgis.com/KYEMegXJrTiWSYWk/arcgis/rest/services"
PAGE_SIZE = 2000

aaf_urls_by_year = {}

# Padrão dinâmico — anos 2010 a 2022 seguem a mesma convenção de nome
for year in range(2010, 2023):
    aaf_urls_by_year[year] = (
        f"{BASE_URL}/db_geo_compartilhado_dmif_fogo_aaf_{year}/FeatureServer/0"
    )

# Exceções — nomes de serviço mudaram ano a ano a partir de 2023
aaf_urls_by_year[2023] = f"{BASE_URL}/AAF_2023_DGEO_ICMBIO_oficial/FeatureServer/0"
aaf_urls_by_year[2024] = f"{BASE_URL}/AAF_2024_DGEO_ICMBIO_oficial/FeatureServer/0"
aaf_urls_by_year[2025] = f"{BASE_URL}/AAF_2025_DGEO_oficial/FeatureServer/0"
aaf_urls_by_year[2026] = f"{BASE_URL}/AAF_2026/FeatureServer/0"

print("Total de anos mapeados:", len(aaf_urls_by_year))

Total de anos mapeados: 17


In [14]:
# Download com paginação, salvando um GeoJSON por ano

aaf_dir = os.path.join(FILES_DIR, "aaf")
os.makedirs(aaf_dir, exist_ok=True)

aaf_downloaded_files = {}

for year, base_url in aaf_urls_by_year.items():

    output_path = os.path.join(aaf_dir, f"aaf_{year}.geojson")

    if os.path.exists(output_path):
        aaf_downloaded_files[year] = output_path
        continue

    # Paginação — segue baixando enquanto o servidor retornar página cheia
    all_features = []
    offset = 0

    while True:
        query_url = (
            f"{base_url}/query?where=1%3D1&outFields=*&outSR=4326&f=geojson"
            f"&resultOffset={offset}&resultRecordCount={PAGE_SIZE}"
        )
        response = requests.get(query_url, timeout=120)
        response.raise_for_status()
        page_data = response.json()

        features = page_data.get("features", [])
        if not features:
            break

        all_features.extend(features)

        if len(features) < PAGE_SIZE:
            break

        offset += PAGE_SIZE

    # Salvar como GeoJSON válido
    final_geojson = {"type": "FeatureCollection", "features": all_features}

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(final_geojson, f)

    aaf_downloaded_files[year] = output_path
    print(f"[{year}] Salvo — {len(all_features)} registros")

print("\nAnos baixados:", list(aaf_downloaded_files.keys()))


Anos baixados: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]


In [18]:
# Carregar todos os anos, checando consistência de colunas antes de concatenar

aaf_gdf_list = []
reference_columns = None

for year, path in aaf_downloaded_files.items():
    gdf_year = gpd.read_file(path)

    if reference_columns is None:
        reference_columns = set(gdf_year.columns)
    else:
        diff = set(gdf_year.columns).symmetric_difference(reference_columns)
        if diff:
            print(f"[{year}] Diferença de colunas em relação ao primeiro ano: {diff}")

    gdf_year["file_year"] = year
    aaf_gdf_list.append(gdf_year)

gdf_aaf = gpd.GeoDataFrame(pd.concat(aaf_gdf_list, ignore_index=True))
gdf_aaf_old = gpd.GeoDataFrame(pd.concat(aaf_gdf_list, ignore_index=True))


print("Shape:", gdf_aaf.shape)
print("Colunas:", gdf_aaf.columns.tolist())
print("Anos presentes:", sorted(gdf_aaf["file_year"].unique()))

[2024] Diferença de colunas em relação ao primeiro ano: {'ngi', 'mes_num', 'ct', 'cr', 'area_uc', 'gr_nome', 'data_img', 'local', 'categoria', 'bioma', 'ano', 'area_ent', 'mes_nome', 'GlobalID', 'data', 'pk_aaf', 'FID'}
[2025] Diferença de colunas em relação ao primeiro ano: {'ngi', 'mes_num', 'Shape__Are', 'ct', 'acao', 'cr', 'area_uc', 'gr_nome', 'Shape__Len', 'data_img', 'local', 'categoria', 'bioma', 'ano', 'tipo', 'area_ent', 'mes_nome', 'GlobalID', 'data', 'pk_aaf', 'classe', 'FID'}
[2026] Diferença de colunas em relação ao primeiro ano: {'ngi', 'mes_num', 'Shape__Are', 'ct', 'acao', 'cr', 'area_uc', 'gr_nome', 'Shape__Len', 'data_img', 'local', 'categoria', 'bioma', 'ano', 'tipo', 'area_ent', 'mes_nome', 'GlobalID', 'data', 'pk_aaf', 'GlobalID_2', 'classe', 'FID'}
Shape: (41780, 33)
Colunas: ['pk_aaf', 'cnuc', 'nome_uc', 'area_ha', 'data_img', 'classe', 'satelite', 'obs', 'juliano', 'Shape__Area', 'Shape__Length', 'geometry', 'file_year', 'FID', 'data', 'ano', 'mes_nome', 'mes_n

### Tratamento dos dados

In [ ]:
# Normalização das colunas de data, sem criar coluna nova, só preenchendo/corrigindo
# as 7 colunas existentes a partir da fonte mais confiável disponível por registro
#
# Prioridade: data_img (mais granular, legado 2011-2023) > data em epoch ms
# (esquema novo 2024-2026) > juliano + file_year (fallback, cobre parte de 2011
# e outros anos esparsos). Quando nenhuma fonte existe (2010 inteiro e parte de
# 2021), as colunas ficam nulas, não tem como reconstruir data que nunca existiu

month_abbreviations = {
    1: "jan", 2: "fev", 3: "mar", 4: "abr", 5: "mai", 6: "jun",
    7: "jul", 8: "ago", 9: "set", 10: "out", 11: "nov", 12: "dez",
}

date_from_img = pd.to_datetime(gdf_aaf["data_img"], errors="coerce")
date_from_epoch = pd.to_datetime(gdf_aaf["data"], unit="ms", errors="coerce")

julian_numeric = pd.to_numeric(gdf_aaf["juliano"], errors="coerce")
julian_numeric = julian_numeric.where(julian_numeric.between(1, 366))
date_from_julian = (
    pd.to_datetime(gdf_aaf["file_year"].astype(str), format="%Y", errors="coerce")
    + pd.to_timedelta(julian_numeric - 1, unit="D")
)

canonical_date = date_from_img.combine_first(date_from_epoch).combine_first(date_from_julian)

gdf_aaf["data_img"] = canonical_date
gdf_aaf["data"] = canonical_date.dt.date
gdf_aaf["ano"] = canonical_date.dt.year
gdf_aaf["mes_num"] = canonical_date.dt.month
gdf_aaf["mes_nome"] = gdf_aaf["mes_num"].map(month_abbreviations)
gdf_aaf["juliano"] = canonical_date.dt.dayofyear

# Tipagem final, Int64 nullable porque ainda sobra NaN nos registros sem nenhuma fonte
gdf_aaf["juliano"] = gdf_aaf["juliano"].astype("Int64")
gdf_aaf["file_year"] = gdf_aaf["file_year"].astype("Int64")
gdf_aaf["ano"] = gdf_aaf["ano"].astype("Int64")
gdf_aaf["mes_num"] = gdf_aaf["mes_num"].astype("Int64")
gdf_aaf["mes_nome"] = gdf_aaf["mes_nome"].astype("string")

print(f"Registros sem nenhuma fonte de data (permanecem nulos): {canonical_date.isna().sum()}")

Registros sem nenhuma fonte de data (permanecem nulos): 585


In [20]:
#  Reprojeção e correção de geometria

gdf_aaf = gdf_aaf.to_crs(METRIC_CRS)
gdf_aaf["geometry"] = gdf_aaf.geometry.buffer(0)

n_invalid = (~gdf_aaf.is_valid).sum()
print(f"Geometrias inválidas após buffer(0): {n_invalid}")
gdf_aaf = gdf_aaf[gdf_aaf.is_valid].copy()

Geometrias inválidas após buffer(0): 2


In [21]:
# Área e perímetro recalculados direto da geometria
# Shape__Area original oscila entre grau² e m² sem padrão fixo por ano
# (confirmado em 2024/2025 vs o resto), não é confiável em nenhum caso

gdf_aaf["area_ha_original"] = gdf_aaf["area_ha"]
gdf_aaf["area_ha"] = gdf_aaf.geometry.area / 10000
gdf_aaf["shape_length"] = gdf_aaf.geometry.length

In [22]:
# Limpeza de string "None" tratada como categoria válida

categorical_columns = ["classe", "acao", "tipo", "categoria"]
for column in categorical_columns:
    gdf_aaf[column] = gdf_aaf[column].replace("None", np.nan)

In [23]:
# Harmonização do tipo de evento
# classe cobre 2020-2024, acao cobre 2025-2026, nenhuma cobre 2010-2019
# tipo e categoria são dimensões diferentes, não entram nessa fusão

harmonization_map = {
    "aceiro": "aceiro",
    "fogo natural": "natural",
    "raio": "natural",
    "gestao de ignicao natural": "natural",
    "incendio": "incendio",
    "indigena": "antropica",
    "gestao de ignicao antropica": "antropica",
    "queima por indigenas isolados": "antropica",
    "outros": "outros",
    "queima controlada": "queima controlada",
    "queima prescrita": "queima prescrita",
}

gdf_aaf["event_type"] = (
    gdf_aaf["classe"]
    .combine_first(gdf_aaf["acao"])
    .map(harmonization_map)
)
gdf_aaf["management_type"] = gdf_aaf["tipo"]
gdf_aaf["response_category"] = gdf_aaf["categoria"]

In [24]:
# Remoção de duplicata geométrica exata

n_before = len(gdf_aaf)
gdf_aaf = gdf_aaf[
    ~gdf_aaf.geometry.apply(lambda geom: geom.wkb).duplicated()
].copy()
print(f"Duplicatas geométricas removidas: {n_before - len(gdf_aaf)}")

Duplicatas geométricas removidas: 283


In [ ]:
# Backfill administrativo via overlay espacial com as UCs
# cnuc, bioma e area_uc são a mesma grandeza nas duas fontes, entram por combine_first

gdf_aaf = gdf_aaf.reset_index(drop=True)
gdf_aaf["event_id"] = gdf_aaf.index

admin_columns = ["cnuc", "bioma", "area_uc"]
needs_backfill = gdf_aaf[admin_columns].isna().any(axis=1)

events_to_backfill = gdf_aaf.loc[needs_backfill, ["event_id", "geometry"]]
uc_columns = ["cnuc", "bioma_pred", "areahaalb", "categoria_"]

overlay_result = gpd.overlay(
    events_to_backfill, gdf_ucs[uc_columns + ["geometry"]], how="intersection"
)
overlay_result["intersection_area"] = overlay_result.geometry.area

best_match = (
    overlay_result
    .sort_values("intersection_area", ascending=False)
    .drop_duplicates(subset="event_id", keep="first")
    .set_index("event_id")
)

gdf_aaf["cnuc"] = gdf_aaf["cnuc"].combine_first(gdf_aaf["event_id"].map(best_match["cnuc"]))
gdf_aaf["bioma"] = gdf_aaf["bioma"].combine_first(gdf_aaf["event_id"].map(best_match["bioma_pred"]))
gdf_aaf["area_uc"] = gdf_aaf["area_uc"].combine_first(gdf_aaf["event_id"].map(best_match["areahaalb"]))


print(f"Eventos com backfill administrativo via overlay: {best_match['cnuc'].notna().sum()} de {needs_backfill.sum()}")

Eventos com backfill administrativo via overlay: 27357 de 28895


c:\Users\ANDERSONALVESCOELHO\miniconda3\envs\geo\Lib\site-packages\geopandas\tools\overlay.py:375: UserWarning: `keep_geom_type=True` in overlay resulted in 9 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = _collection_extract(result, geom_type, keep_geom_type_warning)


In [26]:
# Limpeza final de colunas

dead_columns = ["Shape__Are", "Shape__Len", "GlobalID_2", "Shape__Area", "Shape__Length", "event_id"]
gdf_aaf = gdf_aaf.drop(columns=[c for c in dead_columns if c in gdf_aaf.columns])

print("Shape final:", gdf_aaf.shape)
print("\nDistribuição de event_type:")
print(gdf_aaf["event_type"].value_counts(dropna=False))

Shape final: (41495, 33)

Distribuição de event_type:
event_type
incendio             17076
NaN                  16187
queima prescrita      5743
queima controlada     1697
antropica              371
natural                212
aceiro                 105
outros                 104
Name: count, dtype: int64


### análise e dados inconsistentes

In [33]:
# Comparação de shape e schema, colunas de data, antes vs depois da normalização

date_columns = ["data_img", "data", "ano", "mes_num", "mes_nome", "juliano"]

print(f"gdf_aaf_old (original): {gdf_aaf_old.shape}")
print(f"gdf_aaf (tratado): {gdf_aaf.shape}")

print("\n=== dtype antes vs depois ===")
for column in date_columns:
    dtype_old = gdf_aaf_old[column].dtype if column in gdf_aaf_old.columns else "não existe"
    dtype_new = gdf_aaf[column].dtype if column in gdf_aaf.columns else "não existe"
    print(f"{column:12s} | antes: {str(dtype_old):20s} | depois: {str(dtype_new)}")

gdf_aaf_old (original): (41780, 33)
gdf_aaf (tratado): (41495, 33)

=== dtype antes vs depois ===
data_img     | antes: object               | depois: datetime64[us]
data         | antes: float64              | depois: object
ano          | antes: float64              | depois: Int64
mes_num      | antes: str                  | depois: Int64
mes_nome     | antes: str                  | depois: string
juliano      | antes: object               | depois: Int64


In [36]:
# % vazio por ano, antes vs depois, coluna por coluna
# usa file_year como referência comum, já que é a única coluna estável nas duas versões

pct_null_before = (
    gdf_aaf_old.groupby("file_year")[date_columns]
    .apply(lambda df: df.isna().mean() * 100)
    .round(1)
)

pct_null_after = (
    gdf_aaf.groupby("file_year")[date_columns]
    .apply(lambda df: df.isna().mean() * 100)
    .round(1)
)

for column in date_columns:
    comparison = pd.DataFrame({
        "antes": pct_null_before[column],
        "depois": pct_null_after[column],
    })
    comparison["reducao"] = comparison["antes"] - comparison["depois"]

    print(f"\n=== {column} ===")
    print(comparison)


=== data_img ===
           antes  depois  reducao
file_year                        
2010       100.0   100.0      0.0
2011        95.9     0.1     95.8
2012         0.0     0.0      0.0
2013         0.0     0.0      0.0
2014         0.0     0.0      0.0
2015         0.0     0.0      0.0
2016         0.0     0.0      0.0
2017         0.0     0.0      0.0
2018        13.0    13.1     -0.1
2019         0.0     0.0      0.0
2020         0.0     0.0      0.0
2021         1.5     1.4      0.1
2022         0.0     0.0      0.0
2023         0.0     0.0      0.0
2024       100.0     0.0    100.0
2025       100.0     0.0    100.0
2026       100.0     0.0    100.0

=== data ===
           antes  depois  reducao
file_year                        
2010       100.0   100.0      0.0
2011       100.0     0.1     99.9
2012       100.0     0.0    100.0
2013       100.0     0.0    100.0
2014       100.0     0.0    100.0
2015       100.0     0.0    100.0
2016       100.0     0.0    100.0
2017       100.0

In [37]:
# Faixa de datas (min/max) resultante, antes vs depois
# antes, 'data_img' e 'data' têm formato/unidade misturados, parseia igual
# fizemos no diagnóstico original, pra comparação justa

data_img_before = pd.to_datetime(gdf_aaf_old["data_img"], errors="coerce")
data_epoch_before = pd.to_datetime(gdf_aaf_old["data"], unit="ms", errors="coerce")

print("=== Faixa de datas, ANTES ===")
print(f"data_img (parseada): {data_img_before.min()} até {data_img_before.max()}")
print(f"data (epoch ms, parseada): {data_epoch_before.min()} até {data_epoch_before.max()}")

print("\n=== Faixa de datas, DEPOIS ===")
print(f"data_img (canônica): {gdf_aaf['data_img'].min()} até {gdf_aaf['data_img'].max()}")
print(f"ano: {gdf_aaf['ano'].min()} até {gdf_aaf['ano'].max()}")

=== Faixa de datas, ANTES ===
data_img (parseada): 2011-07-10 00:00:00 até 2023-12-31 00:00:00
data (epoch ms, parseada): 2024-01-01 00:00:00 até 2026-10-18 00:00:00

=== Faixa de datas, DEPOIS ===
data_img (canônica): 2011-04-20 00:00:00 até 2026-10-18 00:00:00
ano: 2011 até 2026


In [38]:
# Amostra lado a lado, mesmos registros antes e depois, pra inspeção visual direta
# usa índice comum, já que gdf_aaf perdeu alguns registros (geometria inválida/nula)

common_indices = gdf_aaf.index.intersection(gdf_aaf_old.index)
sample_indices = (
    pd.Series(common_indices)
    .groupby(gdf_aaf_old.loc[common_indices, "file_year"])
    .apply(lambda s: s.sample(min(2, len(s)), random_state=42))
)
sample_indices = sample_indices.explode().tolist()

side_by_side = pd.concat(
    {
        "antes": gdf_aaf_old.loc[sample_indices, date_columns],
        "depois": gdf_aaf.loc[sample_indices, date_columns],
    },
    axis=1,
)

side_by_side

antes                                                    depois                                           
                  data_img          data     ano mes_num mes_nome juliano   data_img        data   ano mes_num mes_nome juliano
220                   None           NaN     NaN     NaN      NaN    None        NaT         NaT  <NA>    <NA>     <NA>    <NA>
42                    None           NaN     NaN     NaN      NaN    None        NaT         NaT  <NA>    <NA>     <NA>    <NA>
1387                   NaT           NaN     NaN     NaN      NaN     241 2011-07-10  2011-07-10  2011       7      jul     191
663                    NaT           NaN     NaN     NaN      NaN     159 2011-08-10  2011-08-10  2011       8      ago     222
2753   2012-12-03 00:00:00           NaN     NaN     NaN      NaN     338 2012-09-05  2012-09-05  2012       9      set     249
3104   2012-09-21 00:00:00           NaN     NaN     NaN      NaN     265 2012-09-17  2012-09-17  2012       9      set     261
3764   2013-08-24 00:00:00           NaN     NaN     NaN      NaN     236 2013-09-15  2013-09-15  2013       9      set     258
3985   2013-08-07 00:00:00           NaN     NaN     NaN      NaN     219 2013-06-21  2013-06-21  2013       6      jun     172
4881   2014-09-10 00:00:00           NaN     NaN     NaN      NaN     253 2014-07-17  2014-07-17  2014       7      jul     198
5783   2014-06-11 00:00:00           NaN     NaN     NaN      NaN     162 2014-08-28  2014-08-28  2014       8      ago     240
6444   2015-09-06 00:00:00           NaN     NaN     NaN      NaN     249 2015-09-27  2015-09-27  2015       9      set     270
7370   2015-07-05 00:00:00           NaN     NaN     NaN      NaN     186 2015-09-22  2015-09-22  2015       9      set     265
9973   2016-04-11 00:00:00           NaN     NaN     NaN      NaN   101.0 2017-10-18  2017-10-18  2017      10      out     291
9729   2016-08-23 00:00:00           NaN     NaN     NaN      NaN   236.0 2016-08-08  2016-08-08  2016       8      ago     221
11812  2017-09-18 00:00:00           NaN     NaN     NaN      NaN     261 2017-09-03  2017-09-03  2017       9      set     246
12247  2017-09-11 00:00:00           NaN     NaN     NaN      NaN     254 2017-07-24  2017-07-24  2017       7      jul     205
13721  2018-07-03 00:00:00           NaN     NaN     NaN      NaN     184 2018-08-27  2018-08-27  2018       8      ago     239
13338                  NaT           NaN     NaN     NaN      NaN       0        NaT         NaT  <NA>    <NA>     <NA>    <NA>
15367  2019-08-06 00:00:00           NaN     NaN     NaN      NaN     218 2019-09-20  2019-09-20  2019       9      set     263
14903  2019-05-10 00:00:00           NaN     NaN     NaN      NaN     130 2019-08-19  2019-08-19  2019       8      ago     231
18055  2020-07-22 00:00:00           NaN     NaN     NaN      NaN     203 2020-10-06  2020-10-06  2020      10      out     280
17942  2020-09-17 00:00:00           NaN     NaN     NaN      NaN     260 2020-07-06  2020-07-06  2020       7      jul     188
20016  2021-10-03 00:00:00           NaN     NaN     NaN      NaN   276.0 2021-10-07  2021-10-07  2021      10      out     280
19552  2021-07-11 00:00:00           NaN     NaN     NaN      NaN   192.0 2021-07-10  2021-07-10  2021       7      jul     191
22422  2022-10-08 00:00:00           NaN     NaN     NaN      NaN   259.0 2022-05-26  2022-05-26  2022       5      mai     146
21883  2022-09-16 00:00:00           NaN     NaN     NaN      NaN   244.0 2022-10-04  2022-10-04  2022      10      out     277
25670  2023-07-23 00:00:00           NaN     NaN     NaN      NaN     201 2023-08-02  2023-08-02  2023       8      ago     214
27352  2023-10-01 00:00:00           NaN     NaN     NaN      NaN     273 2023-10-16  2023-10-16  2023      10      out     289
31366                  NaN  1.724544e+12  2024.0      08      ago     238 2024-07-31  2024-07-31  2024       7      jul     213
32403                  NaN  1.725581e+12  2024.0     

In [64]:
# Registros removidos no bloco de reprojeção/correção de geometria
# comparação entre gdf_aaf_old (original) e gdf_aaf (já filtrado por is_valid)

removed_indices = gdf_aaf_old.index.difference(gdf_aaf.index)
print(f"Total de registros removidos: {len(removed_indices)}")
print(f"% do total original: {len(removed_indices) / len(gdf_aaf_old) * 100:.3f}%")

removed_records = gdf_aaf_old.loc[
    removed_indices, ["pk_aaf", "file_year", "nome_uc", "cnuc", "area_ha"]
].copy()

print("\n=== Distribuição por ano ===")
print(removed_records["file_year"].value_counts().sort_index())

Total de registros removidos: 285
% do total original: 0.682%

=== Distribuição por ano ===
file_year
2026    285
Name: count, dtype: int64


In [54]:
# Consolidado por ano, só nulo real, que é o que já sabemos que existe
# pra confirmar se a distribuição bate com o que já vimos nos diagnósticos anteriores
# (classe 2020-2024, acao/tipo 2025-2026, categoria 2024-2026)

null_by_year = pd.DataFrame({
    column: gdf_aaf_old.groupby("file_year")[column].apply(lambda s: s.isna().mean() * 100).round(1)
    for column in categorical_columns
})

print(null_by_year)

           classe   acao   tipo  categoria
file_year                                 
2010        100.0  100.0  100.0      100.0
2011        100.0  100.0  100.0      100.0
2012        100.0  100.0  100.0      100.0
2013        100.0  100.0  100.0      100.0
2014         99.8  100.0  100.0      100.0
2015        100.0  100.0  100.0      100.0
2016        100.0  100.0  100.0      100.0
2017         99.9  100.0  100.0      100.0
2018        100.0  100.0  100.0      100.0
2019         96.5  100.0  100.0      100.0
2020          0.0  100.0  100.0      100.0
2021          0.0  100.0  100.0      100.0
2022          0.0  100.0  100.0      100.0
2023          0.0  100.0  100.0      100.0
2024          0.0  100.0  100.0        0.0
2025        100.0    0.0    0.0        0.0
2026        100.0    0.0    0.0        0.0


In [ ]:
# % de duplicata em relação ao volume total de cada ano, pra não confundir
# ano com muita duplicata absoluta com ano que simplesmente tem mais registro

is_duplicate = gdf_aaf_old.geometry.apply(lambda geom: geom.wkb if geom is not None else None).duplicated()

duplicates_by_year = gdf_aaf_old.loc[is_duplicate, "file_year"].value_counts().sort_index()
total_by_year = gdf_aaf_old["file_year"].value_counts().sort_index()

summary = pd.DataFrame({
    "duplicatas": duplicates_by_year,
    "total_registros": total_by_year,
}).fillna(0)
summary["duplicatas"] = summary["duplicatas"].astype(int)
summary["pct_duplicata"] = (summary["duplicatas"] / summary["total_registros"] * 100).round(2)
print(f"Total de duplicatas geométricas: {is_duplicate.sum()}")
summary

Total de duplicatas geométricas: 284


,duplicatas,total_registros,pct_duplicata
file_year,,,
2010,44,354,12.43
2011,18,1583,1.14
2012,17,1628,1.04
2013,0,953,0.00
2014,27,1397,1.93
2015,0,2088,0.00
2016,0,2025,0.00
2017,15,2692,0.56
2018,4,1513,0.26


In [ ]:
# Quantidade e percentual de preenchimento por ano, colunas administrativas
# antes (gdf_aaf_old) e depois (gdf_aaf), lado a lado

from IPython.display import display_html

admin_columns = ["cnuc", "nome_uc", "bioma", "ngi"]
existing_columns = [c for c in admin_columns if c in gdf_aaf.columns]
missing_columns = [c for c in admin_columns if c not in gdf_aaf.columns]

if missing_columns:
    print(f"Colunas que não existem mais em gdf_aaf: {missing_columns}")

counts_before = gdf_aaf_old.groupby("file_year")[admin_columns].apply(lambda df: df.notna().sum())
counts_after = gdf_aaf.groupby("file_year")[existing_columns].apply(lambda df: df.notna().sum())

pct_before = gdf_aaf_old.groupby("file_year")[admin_columns].apply(lambda df: (df.notna().mean() * 100).round(1))
pct_after = gdf_aaf.groupby("file_year")[existing_columns].apply(lambda df: (df.notna().mean() * 100).round(1))

html_count_before = counts_before.to_html().replace("<table", '<table style="display:inline-block; margin-right:40px"')
html_count_after = counts_after.to_html().replace("<table", '<table style="display:inline-block"')

html_pct_before = pct_before.to_html().replace("<table", '<table style="display:inline-block; margin-right:40px"')
html_pct_after = pct_after.to_html().replace("<table", '<table style="display:inline-block"')

display_html(
    f"<h4 style='display:inline-block; margin-right:250px'>Antes (quantidade)</h4><h4 style='display:inline-block'>Depois (quantidade)</h4>"
    f"<br>{html_count_before}{html_count_after}",
    raw=True,
)

display_html(
    f"<h4 style='display:inline-block; margin-right:280px'>Antes (%)</h4><h4 style='display:inline-block'>Depois (%)</h4>"
    f"<br>{html_pct_before}{html_pct_after}",
    raw=True,
)

,cnuc,nome_uc,bioma,ngi
file_year,,,,
2010,352,354,0,0
2011,1583,1583,0,0
2012,1628,1628,0,0
2013,953,953,0,0
2014,1381,1397,0,0
2015,2078,2088,0,0
2016,2010,2025,0,0
2017,2555,2692,0,0
2018,0,1513,0,0


,cnuc,nome_uc,bioma,ngi
file_year,,,,
2010,99.4,100.0,0.0,0.0
2011,100.0,100.0,0.0,0.0
2012,100.0,100.0,0.0,0.0
2013,100.0,100.0,0.0,0.0
2014,98.9,100.0,0.0,0.0
2015,99.5,100.0,0.0,0.0
2016,99.3,100.0,0.0,0.0
2017,94.9,100.0,0.0,0.0
2018,0.0,100.0,0.0,0.0


# MapBiomas Fogo Collection 4 

In [11]:
mapbiomas_fire_annual = ee.Image(
    "projects/mapbiomas-public/assets/brazil/fire/collection4/mapbiomas_fire_collection4_annual_burned_v1"
)

mapbiomas_fire_scar_size = ee.Image(
    "projects/mapbiomas-public/assets/brazil/fire/collection4/mapbiomas_fire_collection4_annual_burned_scar_size_range_v1"
)

print("Bandas (área queimada anual):", mapbiomas_fire_annual.bandNames().getInfo())
print("Bandas (tamanho de cicatriz e frequência):", mapbiomas_fire_scar_size.bandNames().getInfo())

Bandas (área queimada anual): ['burned_area_1985', 'burned_area_1986', 'burned_area_1987', 'burned_area_1988', 'burned_area_1989', 'burned_area_1990', 'burned_area_1991', 'burned_area_1992', 'burned_area_1993', 'burned_area_1994', 'burned_area_1995', 'burned_area_1996', 'burned_area_1997', 'burned_area_1998', 'burned_area_1999', 'burned_area_2000', 'burned_area_2001', 'burned_area_2002', 'burned_area_2003', 'burned_area_2004', 'burned_area_2005', 'burned_area_2006', 'burned_area_2007', 'burned_area_2008', 'burned_area_2009', 'burned_area_2010', 'burned_area_2011', 'burned_area_2012', 'burned_area_2013', 'burned_area_2014', 'burned_area_2015', 'burned_area_2016', 'burned_area_2017', 'burned_area_2018', 'burned_area_2019', 'burned_area_2020', 'burned_area_2021', 'burned_area_2022', 'burned_area_2023', 'burned_area_2024']
Bandas (tamanho de cicatriz e frequência): ['scar_area_ha_1985', 'scar_area_ha_1986', 'scar_area_ha_1987', 'scar_area_ha_1988', 'scar_area_ha_1989', 'scar_area_ha_1990',

# MODIS MCD64A1 

In [ ]:
# MODIS MCD64A1 — leitura prévia via GEE

modis_burned_area = ee.ImageCollection("MODIS/061/MCD64A1").filterDate("2003-01-01", "2025-12-31")

print("Total de imagens:", modis_burned_area.size().getInfo())
print("Bandas:", modis_burned_area.first().bandNames().getInfo())

Total de imagens: 276
Bandas: ['BurnDate', 'Uncertainty', 'QA', 'FirstDay', 'LastDay']


# FireCCI51 (ESA)

In [13]:
# Série parada em 2020-12, não alcança o período recente do AAF, útil só como referência retrospectiva

firecci = ee.ImageCollection("ESA/CCI/FireCCI/5_1")

print("Total de imagens:", firecci.size().getInfo())
print("Bandas:", firecci.first().bandNames().getInfo())

Total de imagens: 240
Bandas: ['BurnDate', 'ConfidenceLevel', 'LandCover', 'ObservedFlag']


# GABAM 

In [15]:
gabam = ee.ImageCollection("projects/sat-io/open-datasets/GABAM")

print("Total de imagens:", gabam.size().getInfo())
print("Bandas:", gabam.first().bandNames().getInfo())

Total de imagens: 14614
Bandas: ['b1']


In [ ]:
# LASA-Alarmes (UFRJ) — sem asset GEE confirmado e sem API pública documentada
# Acesso é via plataforma própria (alarmes.lasa.ufrj.br), possivelmente exige contato direto com o LASA
# Cobertura hoje é regional, V1 beta pra Cerrado/Pantanal e protótipo pra Roraima, não é nacional
# Deixando como placeholder até confirmar via contato com o laboratório

lasa_alarmes_status = "acesso não confirmado — requer contato direto com LASA/UFRJ"

print(lasa_alarmes_status)